# Instalaciones y librerías

In [1]:
!pip install pytorch-lightning
!pip install torchmetrics


In [2]:
import os, glob, re
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import pandas as pd
import shutil

from google.colab import drive

import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pytorch_lightning.loggers import TensorBoardLogger, CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, Callback, EarlyStopping
from torch.nn.modules.loss import SmoothL1Loss

from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from sklearn.linear_model import LinearRegression

from sklearn.model_selection import KFold, StratifiedKFold


# Declaraciones iniciales

In [3]:
# =========================
# CONFIG (edita aquí)
# =========================

drive.mount('/content/drive')

# SPECTRA_DIR    = "/content/drive/MyDrive/HDSP/FIRMAS_A_USAR_solito"  # *.mat
# SPECTRA_DIR    = "/content/drive/MyDrive/HDSP/FIRMAS_RECORTADAS_solito"  # *.mat
SPECTRA_DIR    = "/content/drive/MyDrive/HDSP/FINAL_FIRMAS_MAT"
H5_SUFFIX      = "soil_"                           # patrón nombre
# MAT_SPEC_KEY   = "prom"
#MAT_SPEC_KEY   = "firma_recortada"
MAT_SPEC_KEY   = "data"                      # en cada .mat (220×1)
USE_COLS_1B    = (0, 1)                               # columnas 1-based a promediar → 3..114

# GT_MAT_PATH    = "/content/drive/MyDrive/HDSP/datos_gt_2.mat"
GT_MAT_PATH    = "/content/drive/MyDrive/HDSP/datos_gt_3.mat"
# GT_VAR_NAME    = "gt"
GT_VAR_NAME    = "data"

GT_COLUMN_NAMES = [
    "Numero", "Carbono", "pH", "Ca", "Mg", "Na", "K",
    "Al", "P", "B", "Fe", "Mn", "Cu", "Zn", "CiC", "CE"
]

EXPECTED_SPEC_LEN = 220

TRANSFER_LEARNING = True

PRETRAINED_CKPTS = {
    "pH": "/content/drive/MyDrive/HDSP/MEJORES_MODELOS/pH_best_model.ckpt",
    "Ca": "/content/drive/MyDrive/HDSP/MEJORES_MODELOS/Ca_best_model.ckpt",
}

FREEZE_BACKBONE = True
REINIT_HEAD = True

TARGET_VAR_NAME  = ["pH", "Ca"]
TARGET_VAR_INDEX = None        # o por índice (ej. 2 para pH)

BAND_RANGE     = "ALL"         # "All" | "RE1" | "RE2"
NORMALIZACION  = "ABS_D1_SNV"           # "N" | "SN" | "NZ" | ABS_D1_SNV

#USE_STRATIFIED_KFOLD = True
USE_STRATIFIED_KFOLD = False
STRAT_BINS = 4
CALIBRATE_LINEAR = False

SPLIT_RATIOS   = (0.7, 0.2, 0.1)  # train/val/test
RANDOM_STATE   = 42

BATCH_SIZE     = 4

# max epoch original = 2000
MAX_EPOCHS     = 300
NUM_WORKERS    = 0 # Era 2
RESULT_PATH    = "./resultados_mean_from_GTmat"
SCHEDULER = "plateau"   # "plateau" | "onecycle"
WEIGHT_DECAY = 5e-4     # para AdamW
# base_lr original = 1e-3
BASE_LR = 1e-4
os.makedirs(RESULT_PATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Utilidades varias

In [4]:
# =========================
# Utilidades generales
# =========================

def set_tc_precision():
    try:
        torch.set_float32_matmul_precision("high")  # usa Tensor Cores (RTX)
    except Exception:
        pass

def parse_file_id(path: str):
    base = os.path.basename(path)
    m = re.search(r"([0-9]+){}\.mat$".format(re.escape(H5_SUFFIX)), base)
    if not m:
        m = re.search(r"([0-9]+)", base)
    return int(m.group(1)) if m else None

def list_mat_files(sdir: str):
    files = glob.glob(os.path.join(sdir, "*.mat"))
    return sorted(files, key=lambda p: parse_file_id(p) or 0)

def select_band_slice(T: int, mode: str):
    if mode == "ALL": return slice(0, T)
    if mode == "RE1": return slice(68, 226)
    if mode == "RE2": return slice(288, 449)
    return slice(0, T)

# =========================
# Lectura robusta de .mat (v7 y v7.3)
# =========================
def load_mat_var(path: str, key: str) -> np.ndarray:
    """
    Lee 'key' desde un .mat v7 (scipy) o v7.3 (h5py).
    """
    import scipy.io as sio
    try:
        d = sio.loadmat(path, squeeze_me=True, struct_as_record=False, simplify_cells=True)
        if key not in d:
            raise KeyError(f"Clave '{key}' no encontrada (loadmat) en {path}.")
        arr = np.array(d[key])
    except NotImplementedError:
        import h5py
        with h5py.File(path, "r") as f:
            if key not in f:
                keys = list(f.keys())
                raise KeyError(f"Clave '{key}' no está en {path}. Claves: {keys}")
            arr = f[key][()]

    arr = np.asarray(arr)
    arr = np.squeeze(arr)
    return arr

# =========================
# Cargar mean por archivo desde espectros
# =========================
def load_mean_spectra_per_file():
    files = list_mat_files(SPECTRA_DIR)
    ids, Xs = [], []
    c0 = max(1, USE_COLS_1B[0]) - 1  # 0-based start
    c1 = USE_COLS_1B[1]              # 1-based end (exclusivo)

    for p in files:
        fid = parse_file_id(p)
        if fid is None:
            continue
        R = load_mat_var(p, MAT_SPEC_KEY)
        if R.shape[0] != EXPECTED_SPEC_LEN:
            raise ValueError(f"{p}:{MAT_SPEC_KEY} shape inesperada {R.shape}")
        R_use = R[:]
        #mean_spec = R_use.mean(axis=1).astype(np.float32)  # (512,)
        mean_spec = R_use.astype(np.float32)  # (512,)

        ids.append(fid); Xs.append(mean_spec)

    if not ids:
        raise RuntimeError("No se cargaron espectros. Revisa SPECTRA_DIR / nombres.")
    return np.array(ids, dtype=int), np.stack(Xs, axis=0)

# =========================
# Cargar GT desde datos_gt.mat (gt)
# =========================
def load_gt_from_mat():
    gt = load_mat_var(GT_MAT_PATH, GT_VAR_NAME)  # (N, M)
    if gt.ndim != 2:
        raise ValueError(f"'gt' debe ser 2D, recibido {gt.shape}")
    N, M = gt.shape
    cols = GT_COLUMN_NAMES[:M] if len(GT_COLUMN_NAMES) >= M else \
          (GT_COLUMN_NAMES + [f"Var{j}" for j in range(len(GT_COLUMN_NAMES), M)])
    df = pd.DataFrame(gt, columns=cols[:M])
    return df

def pick_target(y_df: pd.DataFrame, target_name):
    if target_name not in y_df.columns:
        raise KeyError(f"La variable '{target_name}' no está en las columnas del GT.")
    # Extrayendo solo la columna que toca en el bucle del main
    y = y_df[target_name].values.astype(np.float32).reshape(-1, 1)
    return y, target_name

def make_regression_bins(y, n_bins=10):
    y = np.asarray(y).reshape(-1)
    q = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y, q)
    edges = np.unique(edges)
    if len(edges) <= 2:
        return np.zeros_like(y, dtype=int)
    return np.clip(np.digitize(y, edges[1:-1], right=False), 0, len(edges) - 2)


def make_bin_weights(y, n_bins=10):
    bins = make_regression_bins(y, n_bins=n_bins)
    counts = np.bincount(bins)
    counts[counts == 0] = 1
    inv = 1.0 / counts[bins]
    inv = inv / inv.mean()
    return bins, inv.astype(np.float32)

def fit_linear_calibration(y_true, y_pred):
    reg = LinearRegression()
    reg.fit(y_pred.reshape(-1, 1), y_true.reshape(-1, 1))
    return reg


def apply_linear_calibration(model, y_pred):
    return model.predict(y_pred.reshape(-1, 1)).reshape(-1)


 # =========================
 # Gráficas de histogramas de train, val y test
 # =========================

def plot_split_distributions(y_train, y_val, y_test, target_name, fold_num, out_dir):
    """
    Crea un histograma triple para comparar las distribuciones de los conjuntos.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
    data_list = [y_train, y_val, y_test]
    titles = ['Train', 'Validation', 'Test (Externo)']
    colors = ['#1f77b4', '#2ca02c', '#d62728']

    for i, (data, title, color) in enumerate(zip(data_list, titles, colors)):
        axes[i].hist(data.flatten(), bins=20, color=color, alpha=0.7, edgecolor='black')
        axes[i].set_title(f"{title} - {target_name}")
        axes[i].set_xlabel("Valor")
        axes[i].set_ylabel("Frecuencia")

    plt.suptitle(f"Distribución de Datos - {target_name} (Fold {fold_num})", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Guardar la imagen en la carpeta del fold actual
    plt.savefig(os.path.join(out_dir, f"histogramas_distribucion_fold_{fold_num}.jpeg"), dpi=200)
    plt.close()

### Preprocesamiento

In [5]:
# =========================
# Preprocesamiento
# =========================
def normalize_minmax_fit(X_train):
    xmin = X_train.min(); xmax = X_train.max()
    return xmin, xmax

def normalize_minmax_apply(X, xmin, xmax):
    return (X - xmin) / (xmax - xmin + 1e-12)

def zscore_fit(X_train):
    mu = X_train.mean(axis=0, keepdims=True)
    sd = X_train.std(axis=0, keepdims=True) + 1e-12
    return mu, sd

def zscore_apply(X, mu, sd):
    return (X - mu) / sd

def to_absorbance(X):
    """
    Convierte Reflectancia (X) a Absorbancia (Log 1/R).
    Usa 'clip' para asegurar que todos los valores sean estrictamente positivos.
    """
    # np.clip garantiza que X_clipped nunca sea menor que epsilon
    # Esto elimina el 'RuntimeWarning' y los valores NaN.
    X_clipped = np.clip(X, a_min=1e-6, a_max=None)

    # Aplicamos la transformación log(1/R)
    return np.log10(1.0 / X_clipped)

def savgol_smooth(X, window=31, poly=2, deriv=1):
    """
    X: (N, T)
    Aplica Savitzky–Golay a cada espectro (eje 1).

    window: Ventana de suavizado (debe ser impar).
    poly: Orden del polinomio.
    deriv: 0=suavizado, 1=primera derivada, 2=segunda derivada.

    """
    return savgol_filter(X, window_length=window, polyorder=poly, deriv=deriv, axis=1)

def snv(X):
    """
    Standard Normal Variate (SNV) por muestra:
    para cada fila i: (x_i - mean_i) / std_i
    """
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True)
    return (X - mean) / (std + 1e-12)

### CNN

In [6]:
# =========================
# Dataset y Modelo (igual a tu código actual)
# =========================
class SpectraDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N,1,T)
        self.y = torch.tensor(y, dtype=torch.float32)
        #self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # (N,1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class ResBlock(nn.Module):
    def __init__(self, channels: int, p: float = 0.1, dilation: int = 1): # Original era p=0.1 luego cambié a p = 0.1
        super().__init__()
        # padding = dilation para mantener longitud (same length)
        self.conv1 = nn.Conv1d(
            channels, channels, kernel_size=3,
            padding=dilation, dilation=dilation, bias=False
        )
        self.bn1 = nn.BatchNorm1d(channels)

        self.conv2 = nn.Conv1d(
            channels, channels, kernel_size=3,
            padding=dilation, dilation=dilation, bias=False
        )
        self.bn2 = nn.BatchNorm1d(channels)

        self.dropout = nn.Dropout(p)

    def forward(self, x):
        identity = x
        xo = self.conv1(x)
        xo = self.bn1(xo)
        xo = torch.relu(xo)

        xo = self.dropout(xo)

        xo = self.conv2(xo)
        xo = self.bn2(xo)

        xo = xo + identity
        out = torch.relu(xo)
        return out

class Conv1DRegressor(pl.LightningModule):
    def __init__(self, input_length: int):
        super().__init__()
        self.save_hyperparameters()

        self.conv_stem = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3, bias=False), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2, bias=False), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
        )

        # Bloques residuales (mantienen longitud: padding=dilation)
        self.res = nn.Sequential(
            ResBlock(64, p=0.10, dilation=1),
            ResBlock(64, p=0.10, dilation=2),  # campo receptivo mayor
        )

        self.pool = nn.AdaptiveAvgPool1d(8)
        self.flatten = nn.Flatten()

        self.fc = nn.Sequential(
            nn.Linear(64 * 8, 64),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(64, 1),
        )

        self.crit = nn.SmoothL1Loss()

    def forward(self, x):
        x = self.conv_stem(x)
        x = self.res(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x


    def on_validation_epoch_start(self):
        self.vp, self.vt = [], []

    def validation_step(self, batch, _):
        x, y = batch
        yhat = self(x)
        val_loss = self.crit(yhat, y)
        self.log("val_mse", val_loss, prog_bar=True, on_step=False, on_epoch=True, sync_dist=False)
        #self.log("val_mse", val_mse, prog_bar=True, on_epoch=True, sync_dist=False)

        self.vp.append(yhat.detach().cpu())
        self.vt.append(y.detach().cpu())

        return val_loss

    def on_validation_epoch_end(self):
        if len(self.vp) > 0:
            yp = torch.cat(self.vp).numpy().squeeze()
            yt = torch.cat(self.vt).numpy().squeeze()

            if len(np.unique(yt)) > 1:
                val_r2 = float(r2_score(yt, yp))
                self.log("val_r2", val_r2, prog_bar=True, on_step=False, on_epoch=True, sync_dist=False)

            self.vp.clear()
            self.vt.clear()

    def on_train_epoch_start(self):
        self.tp, self.tt = [], []

    def training_step(self, batch, _):
        x, y = batch
        yhat = self(x)
        loss = self.crit(yhat, y)

        self.log("train_mse", loss, prog_bar=True, on_step=False, on_epoch=True, sync_dist=False)

        self.tp.append(yhat.detach().cpu())
        self.tt.append(y.detach().cpu())

        return loss

    def on_train_epoch_end(self):
        if len(self.tp) > 0:
            yp = torch.cat(self.tp).numpy().squeeze()
            yt = torch.cat(self.tt).numpy().squeeze()

            if len(np.unique(yt)) > 1:
                train_r2 = float(r2_score(yt, yp))
                self.log("train_r2", train_r2, prog_bar=False, on_step=False, on_epoch=True, sync_dist=False)

            self.tp.clear()
            self.tt.clear()

    def configure_optimizers(self):
    # Modo debug: sin scheduler, sin weight_decay
        #return torch.optim.Adam(self.parameters(), lr=1e-3)

        trainable_params = [p for p in self.parameters() if p.requires_grad]

        return torch.optim.Adam(trainable_params, lr=BASE_LR, weight_decay=WEIGHT_DECAY)


def build_transfer_model(target_name, input_length):
    """
    Construye un modelo nuevo para las firmas actuales y transfiere
    las capas convolucionales desde un modelo preentrenado.
    """
    model = Conv1DRegressor(input_length=input_length)

    if not TRANSFER_LEARNING:
        return model

    ckpt_path = PRETRAINED_CKPTS[target_name]

    old_model = Conv1DRegressor.load_from_checkpoint(
        ckpt_path,
        input_length=input_length,
        map_location="cpu"
    )

    # Transferir solamente el extractor de características
    model.conv_stem.load_state_dict(old_model.conv_stem.state_dict())
    model.res.load_state_dict(old_model.res.state_dict())

    # La cabeza fully connected queda nueva
    # Esto es recomendable porque ahora tienes solo 20 muestras
    # y posiblemente otra distribución de datos.
    if not REINIT_HEAD:
        model.fc.load_state_dict(old_model.fc.state_dict())

    if FREEZE_BACKBONE:
        for p in model.conv_stem.parameters():
            p.requires_grad = False

        for p in model.res.parameters():
            p.requires_grad = False

    return model

### Callback y plots

In [7]:
# =========================
# Callback para métricas por época (train/val/test) y plots
# =========================

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    preds, trues = [], []

    device = next(model.parameters()).device

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        yhat = model(xb)

        preds.append(yhat.detach().cpu().numpy())
        trues.append(yb.detach().cpu().numpy())

    y_pred = np.concatenate(preds, axis=0).squeeze()
    y_true = np.concatenate(trues, axis=0).squeeze()
    return y_true, y_pred


def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else float("nan")

    return {
        "mse": float(mse),
        "rmse": float(rmse),
        "mae": float(mae),
        "r2": float(r2),
    }


def save_curves_from_logger(metrics_csv_path, out_dir):
    if not os.path.exists(metrics_csv_path):
        print(f"No se encontró metrics.csv en: {metrics_csv_path}")
        return

    dfm = pd.read_csv(metrics_csv_path)
    print("Columnas en metrics.csv:", dfm.columns.tolist())

    if "epoch" not in dfm.columns:
        print("No se encontró columna 'epoch'")
        return

    # Agrupar por epoch y tomar el último valor no nulo de cada época
    df_epoch = dfm.groupby("epoch", as_index=False).last()

    # -------- curva MSE --------
    plt.figure(figsize=(8, 5))
    plotted = False

    if "train_mse" in df_epoch.columns:
        d = df_epoch[["epoch", "train_mse"]].dropna()
        if len(d) > 0:
            plt.plot(d["epoch"], d["train_mse"], label="Train SmoothL1")
            plotted = True

    if "val_mse" in df_epoch.columns:
        d = df_epoch[["epoch", "val_mse"]].dropna()
        if len(d) > 0:
            plt.plot(d["epoch"], d["val_mse"], label="Val SmoothL1")
            plotted = True

    if plotted:
        plt.xlabel("Época")
        plt.ylabel("SmoothL1")
        plt.title("Curva de pérdida (SmoothL1)")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "curva_loss.jpeg"), dpi=200)
    else:
        print("No se pudo construir curva_loss.jpeg")
    plt.close()

    # -------- curva R2 --------
    plt.figure(figsize=(8, 5))
    plotted = False

    if "train_r2" in df_epoch.columns:
        d = df_epoch[["epoch", "train_r2"]].dropna()
        if len(d) > 0:
            plt.plot(d["epoch"], d["train_r2"], label="Train R²")
            plotted = True

    if "val_r2" in df_epoch.columns:
        d = df_epoch[["epoch", "val_r2"]].dropna()
        if len(d) > 0:
            plt.plot(d["epoch"], d["val_r2"], label="Val R²")
            plotted = True

    if plotted:
        plt.xlabel("Época")
        plt.ylabel("R²")
        plt.title("Curva de precisión (R²)")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "curva_r2.jpeg"), dpi=200)
    else:
        print("No se pudo construir curva_r2.jpeg")
    plt.close()


def save_fold_scatter(y, yhat, target_to_train, fold, name, filename, OUT):
    plt.figure(figsize=(6,5))
    plt.scatter(y, yhat, alpha=0.5, s=15)
    m, M = float(np.min(y)), float(np.max(y))
    plt.plot([m, M], [m, M], 'r--')
    plt.title(f"{target_to_train} - {name} (Fold {fold})")
    plt.xlabel("Real")
    plt.ylabel("Predicho")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, filename), dpi=150)
    plt.close()

### Main (pH y Calcio)

In [8]:
if __name__ == "__main__":
    set_tc_precision()

    ids_spec, X_full_raw = load_mean_spectra_per_file()
    gt_df = load_gt_from_mat()

    # Ordenar las firmas como soil_1, soil_2, ..., soil_20
    order = np.argsort(ids_spec)
    ids_spec = ids_spec[order]
    X_full_base = X_full_raw[order, :]

    # En esta versión, el emparejamiento se hace por orden:
    # soil_1 -> fila 0 del GT, soil_2 -> fila 1 del GT, ..., soil_20 -> fila 19 del GT
    gt_df = gt_df.reset_index(drop=True)

    if len(gt_df) != len(X_full_base):
        raise ValueError(
            f"No coincide el número de firmas ({len(X_full_base)}) "
            f"con el número de filas del GT ({len(gt_df)})."
        )

    common_ids = ids_spec

    # --- Bucle por cada variable objetivo ---
    for target_to_train in ["pH", "Ca"]:
        print(f"\n\n" + "=========================")
        print(f" PROCESANDO VARIABLE: {target_to_train}")
        print("=========================" + "\n")

        # ---------------------------------------------------------
        # 1) Preparar y_full específico para esta variable
        # ---------------------------------------------------------
        y_all_df = gt_df.copy()
        y_full, target_name = pick_target(y_all_df, target_name=target_to_train)

        # ---------------------------------------------------------
        # 2) Selección de bandas (se mantiene igual)
        # ---------------------------------------------------------
        T = X_full_base.shape[1]
        band_sl = select_band_slice(T, BAND_RANGE)
        X_current = X_full_base[:, band_sl]

        # ---------------------------------------------------------
        # 3) Preparando split del kfold
        # ---------------------------------------------------------
        indices = np.arange(len(common_ids))
        np.random.seed(RANDOM_STATE)
        np.random.shuffle(indices)

        split_test = int(0.8 * len(indices))
        idx_train_val_full = indices[:split_test]
        idx_test = indices[split_test:]

        X_te_raw = X_current[idx_test]
        y_te_raw = y_full[idx_test]

        # ---------------------------------------------------------
        # 4) Preparando kfold
        # ---------------------------------------------------------

        k_folds = 3

        y_train_val_full = y_full[idx_train_val_full].reshape(-1)
        bins_train_val_full = make_regression_bins(y_train_val_full, n_bins=STRAT_BINS)
        if USE_STRATIFIED_KFOLD and len(np.unique(bins_train_val_full)) > 1:
            kf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=RANDOM_STATE)
            split_iter = kf.split(idx_train_val_full, bins_train_val_full)
        else:
            kf = KFold(n_splits=k_folds, shuffle=True, random_state=RANDOM_STATE)
            split_iter = kf.split(idx_train_val_full)

        fold_results = []     # Para guardar métricas de cada fold

        for fold, (train_idx, val_idx) in enumerate(split_iter, start=1):

            print(f"\n>>> {target_to_train} - FOLD {fold}/{k_folds}")

            X_tr_raw = X_current[idx_train_val_full[train_idx]]
            y_tr_raw = y_full[idx_train_val_full[train_idx]]

            X_va_raw = X_current[idx_train_val_full[val_idx]]
            y_va_raw = y_full[idx_train_val_full[val_idx]]


            # ---------------------------------------------------------
            # 5) Preprocesamiento de X
            # ---------------------------------------------------------

            if NORMALIZACION == "N":
                xmin, xmax = normalize_minmax_fit(X_tr_raw)
                X_tr = normalize_minmax_apply(X_tr_raw, xmin, xmax)
                X_va = normalize_minmax_apply(X_va_raw, xmin, xmax)
                X_te = normalize_minmax_apply(X_te_raw, xmin, xmax)
            elif NORMALIZACION == "SN":
                X_tr, X_va, X_te = X_tr_raw, X_va_raw, X_te_raw
            elif NORMALIZACION == "NZ":
                xmin, xmax = normalize_minmax_fit(X_tr_raw)
                X_tr_n = normalize_minmax_apply(X_tr_raw, xmin, xmax)
                X_va_n = normalize_minmax_apply(X_va_raw, xmin, xmax)
                X_te_n = normalize_minmax_apply(X_te_raw, xmin, xmax)
                mu, sd = zscore_fit(X_tr_n)
                X_tr = zscore_apply(X_tr_n, mu, sd)
                X_va = zscore_apply(X_va_n, mu, sd)
                X_te = zscore_apply(X_te_n, mu, sd)

            elif NORMALIZACION == "ABS_D1_SNV":

                # 1. Convertir Train, Val y Test a Absorbancia
                X_tr_abs = to_absorbance(X_tr_raw)
                X_va_abs = to_absorbance(X_va_raw)
                X_te_abs = to_absorbance(X_te_raw)

                # 2. Calcular 1ra Derivada (elimina línea base)
                X_tr_d1 = savgol_smooth(X_tr_abs)
                X_va_d1 = savgol_smooth(X_va_abs)
                X_te_d1 = savgol_smooth(X_te_abs)

                # 3. Aplicar SNV (normaliza brillo)
                X_tr = snv(X_tr_d1)
                X_va = snv(X_va_d1)
                X_te = snv(X_te_d1)

            else:
                raise ValueError(f"Modo de NORMALIZACION desconocido: {NORMALIZACION}")

            # ---------------------------------------------------------
            # 5.5) Random Forest y PLSR
            # ---------------------------------------------------------

            #Xtr_flat = X_tr.reshape(X_tr.shape[0], -1)
            #Xva_flat = X_va.reshape(X_va.shape[0], -1)
            #Xte_flat = X_te.reshape(X_te.shape[0], -1)

            #ytr_classic = y_tr_raw.reshape(-1)
            #yva_classic = y_va_raw.reshape(-1)
            #yte_classic = y_te_raw.reshape(-1)

            #baseline_results = {}

            # ------------------------
            # RANDOM FOREST
            # ------------------------
            #rf = RandomForestRegressor(
                #n_estimators=300,
                #max_depth=12,
                #min_samples_split=4,
                #min_samples_leaf=2,
                #random_state=RANDOM_STATE,
                #n_jobs=-1,
            #)

            #rf.fit(Xtr_flat, ytr_classic)

            #ytr_rf = rf.predict(Xtr_flat)
            #yva_rf = rf.predict(Xva_flat)
            #yte_rf = rf.predict(Xte_flat)

            #baseline_results["RF"] = {
                #"train": regression_metrics(ytr_classic, ytr_rf),
                #"val": regression_metrics(yva_classic, yva_rf),
                #"test": regression_metrics(yte_classic, yte_rf),
            #}

            #print(f"[{target_to_train}][RF]   "
                  #f"Train R2={baseline_results['RF']['train']['r2']:.4f} | "
                  #f"Val R2={baseline_results['RF']['val']['r2']:.4f} | "
                  #f"Test R2={baseline_results['RF']['test']['r2']:.4f}")

            # ------------------------
            # PLSR
            # ------------------------
            #plsr = Pipeline([
                #("scaler", StandardScaler()),
                #("pls", PLSRegression(n_components=10))
            #])

            #plsr.fit(Xtr_flat, ytr_classic)

            #ytr_pls = plsr.predict(Xtr_flat).reshape(-1)
            #yva_pls = plsr.predict(Xva_flat).reshape(-1)
            #yte_pls = plsr.predict(Xte_flat).reshape(-1)


            #baseline_results["PLSR"] = {
                #"train": regression_metrics(ytr_classic, ytr_pls),
                #"val": regression_metrics(yva_classic, yva_pls),
                #"test": regression_metrics(yte_classic, yte_pls),
            #}

            #print(f"[{target_to_train}][PLSR] "
                  #f"Train R2={baseline_results['PLSR']['train']['r2']:.4f} | "
                  #f"Val R2={baseline_results['PLSR']['val']['r2']:.4f} | "
                  #f"Test R2={baseline_results['PLSR']['test']['r2']:.4f}")

            # ---------------------------------------------------------
            # 6) Escalado de y (fit SOLO en train)
            # ---------------------------------------------------------

            y_scaler = StandardScaler()

            y_tr = y_scaler.fit_transform(y_tr_raw.reshape(-1, 1)).astype(np.float32)
            y_va = y_scaler.transform(y_va_raw.reshape(-1, 1)).astype(np.float32)
            y_te = y_scaler.transform(y_te_raw.reshape(-1, 1)).astype(np.float32)

            # ---------------------------------------------------------
            # 7) DataLoaders y Modelo
            # ---------------------------------------------------------
            ds_tr = SpectraDataset(X_tr, y_tr)
            ds_va = SpectraDataset(X_va, y_va)
            ds_te = SpectraDataset(X_te, y_te)

            tl = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
            tl_eval = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
            vl = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
            te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

            model = build_transfer_model(target_name=target_to_train, input_length=X_tr.shape[1])

            # ---------------------------------------------------------
            # 8) Configurar Logger con el nombre de la variable
            # ---------------------------------------------------------
            exp = f"KFold_{target_to_train}_{NORMALIZACION}"
            tb_logger = TensorBoardLogger(save_dir=RESULT_PATH, name=exp, version=f"fold_{fold}")
            csv_logger = CSVLogger(save_dir=RESULT_PATH, name=exp, version=f"fold_{fold}")
            OUT = tb_logger.log_dir; os.makedirs(OUT, exist_ok=True)

            # ---------------------------------------------------------
            # 9) Creación de histogramas de train, val y test
            # ---------------------------------------------------------

            # plot_split_distributions(y_tr, y_va, y_te, target_to_train, fold + 1, OUT)

            # ---------------------------------------------------------
            # 10) Entrenamiento: callbacks y trainer
            # ---------------------------------------------------------

            #metrics_cb = EpochMetricsCallback(tl, vl, te, OUT)

            ckpt_cb = ModelCheckpoint(
                dirpath=OUT,
                monitor="val_mse",
                mode="min",
                save_top_k=1,
                filename="best_model",
            )

            early_stop_cb = EarlyStopping(
                monitor="val_mse",
                mode="min",
                patience=15,
            )

            trainer = pl.Trainer(
                max_epochs=MAX_EPOCHS,
                logger=[tb_logger, csv_logger],
                callbacks=[
                    ckpt_cb,
                    early_stop_cb,
                    LearningRateMonitor(logging_interval="epoch"),
                    #LearningRateMonitor(logging_interval="step"),
                    #metrics_cb
                ],
                accelerator="gpu" if torch.cuda.is_available() else "cpu",
                devices=1,
                log_every_n_steps=10
            )

            trainer.fit(model, tl, vl)

            # ---------------------------------------------------------
            # 10) Cargar el mejor modelo de esta variable (Ca o pH)
            # ---------------------------------------------------------
            best_model_path = ckpt_cb.best_model_path
            print(f"[{target_to_train} - Fold {fold}] Cargando mejor modelo: {best_model_path}")

            if best_model_path:
                map_loc = "cuda" if torch.cuda.is_available() else "cpu"
                # Cargamos el modelo específico que se acaba de guardar
                model = Conv1DRegressor.load_from_checkpoint(
                    best_model_path,
                    input_length=X_tr.shape[1],
                    map_location=map_loc,
                ).to(map_loc).eval()
            else:
                print(f"Advertencia: No se encontró checkpoint para {target_to_train}")

            # ---------------------------------------------------------
            # 11) Predicciones en escala normalizada
            # ---------------------------------------------------------
            y_tr_true_scaled, yhat_tr_scaled = predict_loader(model, tl_eval)
            #y_tr_true_scaled, yhat_tr_scaled = predict_loader(model, tl)
            y_va_true_scaled, yhat_va_scaled = predict_loader(model, vl)
            y_te_true_scaled, yhat_te_scaled = predict_loader(model, te)

            # ---------------------------------------------------------
            # 12) Volver a escala original
            # ---------------------------------------------------------

            y_tr_np = y_scaler.inverse_transform(y_tr_true_scaled.reshape(-1, 1)).squeeze()
            y_va_np = y_scaler.inverse_transform(y_va_true_scaled.reshape(-1, 1)).squeeze()
            y_te_np = y_scaler.inverse_transform(y_te_true_scaled.reshape(-1, 1)).squeeze()

            yhat_tr = y_scaler.inverse_transform(yhat_tr_scaled.reshape(-1, 1)).squeeze()
            yhat_va = y_scaler.inverse_transform(yhat_va_scaled.reshape(-1, 1)).squeeze()
            yhat_te = y_scaler.inverse_transform(yhat_te_scaled.reshape(-1, 1)).squeeze()

            #if CALIBRATE_LINEAR:
            #    calib = fit_linear_calibration(y_va_np, yhat_va)
            #    yhat_tr = apply_linear_calibration(calib, yhat_tr)
            #    yhat_va = apply_linear_calibration(calib, yhat_va)
            #    yhat_te = apply_linear_calibration(calib, yhat_te)

            # ---------------------------------------------------------
            # 13) Métricas
            # ---------------------------------------------------------

            metrics_tr = regression_metrics(y_tr_np, yhat_tr)
            metrics_va = regression_metrics(y_va_np, yhat_va)
            metrics_te = regression_metrics(y_te_np, yhat_te)

            print(f"Train: {metrics_tr}")
            print(f"Val:   {metrics_va}")
            print(f"Test:  {metrics_te}")

            fold_results.append({
                "fold": fold,
                "train_r2": metrics_tr["r2"],
                "val_r2": metrics_va["r2"],
                "test_r2": metrics_te["r2"],
                "train_rmse": metrics_tr["rmse"],
                "val_rmse": metrics_va["rmse"],
                "test_rmse": metrics_te["rmse"],
                "train_mae": metrics_tr["mae"],
                "val_mae": metrics_va["mae"],
                "test_mae": metrics_te["mae"],

                #"rf_r2_train": baseline_results["RF"]["train"]["r2"],
                #"rf_r2_val": baseline_results["RF"]["val"]["r2"],
                #"rf_r2_test": baseline_results["RF"]["test"]["r2"],

                #"plsr_r2_train": baseline_results["PLSR"]["train"]["r2"],
                #"plsr_r2_val": baseline_results["PLSR"]["val"]["r2"],
                #"plsr_r2_test": baseline_results["PLSR"]["test"]["r2"],
            })


            print("Train real min/max:", np.min(y_tr_np), np.max(y_tr_np))
            print("Train pred min/max:", np.min(yhat_tr), np.max(yhat_tr))
            print("Val   real min/max:", np.min(y_va_np), np.max(y_va_np))
            print("Val   pred min/max:", np.min(yhat_va), np.max(yhat_va))
            print("Test  real min/max:", np.min(y_te_np), np.max(y_te_np))
            print("Test  pred min/max:", np.min(yhat_te), np.max(yhat_te))

            print("Ejemplo train real:", y_tr_np[:10])
            print("Ejemplo train pred:", yhat_tr[:10])

            # ---------------------------------------------------------
            # 13) Guardado de Archivos (Separados por carpeta de variable)
            # ---------------------------------------------------------

            # Guardar en .mat
            sio.savemat(os.path.join(OUT, f"predicciones_fold_{fold}.mat"), {
                "y_train": y_tr_np,
                "yhat_train": yhat_tr,
                "y_val": y_va_np,
                "yhat_val": yhat_va,
                "y_test": y_te_np,
                "yhat_cnn": yhat_te,
                #"yhat_rf": yte_rf
            })

            metrics_csv_path = os.path.join(csv_logger.log_dir, "metrics.csv")
            save_curves_from_logger(metrics_csv_path, OUT)

            # ---------------------------------------------------------
            # 14) Gráficos Scatter (Save Scatter)
            # ---------------------------------------------------------

            save_fold_scatter(y_tr_np, yhat_tr, target_to_train, fold, "CNN Train", f"scatter_cnn_train_fold_{fold}.jpeg", OUT)
            save_fold_scatter(y_va_np, yhat_va, target_to_train, fold, "CNN Val",   f"scatter_cnn_val_fold_{fold}.jpeg", OUT)
            save_fold_scatter(y_te_np, yhat_te, target_to_train, fold, "CNN Test",  f"scatter_cnn_fold_{fold}.jpeg", OUT)


            # ---------------------------------------------------------
            # 15) Limpieza de memoria (Vital en K-Fold)
            # ---------------------------------------------------------
            del model, trainer, ds_tr, ds_va, ds_te, tl, vl, te, tl_eval
            torch.cuda.empty_cache()


        print(f"Variable {target_to_train} completada. Resultados en: {OUT}")

        print(f"=========================")
        print(f" RESUMEN FINAL PARA: {target_to_train}")

        #avg_rf_test = np.mean([f["rf_r2_test"] for f in fold_results])
        #avg_plsr_test = np.mean([f["plsr_r2_test"] for f in fold_results])


        avg_train = np.mean([f["train_r2"] for f in fold_results])
        avg_val = np.mean([f["val_r2"] for f in fold_results])
        avg_test = np.mean([f["test_r2"] for f in fold_results])
        #print(f"R2 RF (Promedio 5-folds): {avg_rf:.4f}")
        print(f"R2 CNN Train (Promedio {k_folds}-folds): {avg_train:.4f}")
        print(f"R2 CNN Val   (Promedio {k_folds}-folds): {avg_val:.4f}")
        print(f"R2 CNN Test  (Promedio {k_folds}-folds): {avg_test:.4f}")

        #print(f"R2 RF Test    (Promedio {k_folds}-folds): {avg_rf_test:.4f}")
        #print(f"R2 PLSR Test  (Promedio {k_folds}-folds): {avg_plsr_test:.4f}")

        print(f"=========================")

    print("\n PROCESO COMPLETO PARA TODAS LAS VARIABLES.")



 PROCESANDO VARIABLE: pH


>>> pH - FOLD 1/3


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning_fabric/loggers/csv_logs.py:268: Experiment logs directory ./resultados_mean_from_GTmat/KFold_pH_ABS_D1_SNV/fold_1 exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /content/resultados_mean_from_GTmat/KFold_pH_ABS_D1_SNV/fold_1 exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv_stem │ Sequential        │ 10.7 K │ train │     0 │
│ 1 │ res       │ Sequential        │ 49.7 K │ train │     0 │
│ 2 │ pool      │ AdaptiveAvgPool1d │      0 │ train │     0 │
│ 3 │ flatten   │ Flatten           │      0 │ train │     0 │
│ 4 │ fc        │ Sequential        │ 32.9 K │ train │     0 │
│ 5 │ crit      │ SmoothL1Loss      │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 32.9 K                                                                                           
Non-trainable params: 60.3 K                                                                                       
Total params: 93.2 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:317: The number of training batches (3)
is smaller than the logging interval Trainer(log_every_n_steps=10). Set a lower value for log_every_n_steps if you 
want to see logs for the training epoch.

[Ca - Fold 3] Cargando mejor modelo: /content/resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/best_model.ckpt
Train: {'mse': 111.46977233886719, 'rmse': 10.557924622711946, 'mae': 7.62558126449585, 'r2': -0.12097382545471191}
Val:   {'mse': 299.50762939453125, 'rmse': 17.306288723886794, 'mae': 13.26390552520752, 'r2': -0.22779309749603271}
Test:  {'mse': 19.27705955505371, 'rmse': 4.390564833259351, 'mae': 3.7452120780944824, 'r2': 0.08428740501403809}
Train real min/max: 0.9299998 27.9
Train pred min/max: 4.9630866 7.420054
Val   real min/max: 0.9200001 36.8
Val   pred min/max: 4.731584 7.254619
Test  real min/max: 1.94 14.100001
Test  pred min/max: 5.7502255 6.75679
Ejemplo train real: [26.         5.34       2.58       1.29      20.3        3.
  1.1100006  4.77       4.01       0.9299998]
Ejemplo train pred: [4.9630866 5.9303675 6.735621  7.328219  6.356809  7.420054  5.6771426
 5.900675  5.250406  6.127235 ]
Columnas en metrics.csv: ['epoch', 'lr-Adam', 'step', 'train_mse', 

In [9]:
!zip -r resultados_mean_from_GTmat.zip resultados_mean_from_GTmat


  adding: resultados_mean_from_GTmat/ (stored 0%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/ (stored 0%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/ (stored 0%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/curva_loss.jpeg (deflated 52%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/scatter_cnn_train_fold_3.jpeg (deflated 47%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/metrics.csv (deflated 54%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/events.out.tfevents.1777700526.f32092a3f797.3781.5 (deflated 63%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/scatter_cnn_fold_3.jpeg (deflated 48%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/best_model.ckpt (deflated 8%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/hparams.yaml (stored 0%)
  adding: resultados_mean_from_GTmat/KFold_Ca_ABS_D1_SNV/fold_3/curva_r2.jpeg (deflated 49%)